# Module 4 · Pre-training Objectives & BERT

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 04 (Seq2Seq & Attention), DL 07 (Transformer Architecture)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 120–150 minutes

---



## 🎓 Socratic Deep Check

> *"In Word2Vec, the word 'bank' has one vector. In BERT, it has thousands. How does predicting the context around a [MASK] token enable a model to learn that 'river bank' and 'bank deposit' are different concepts? Does the model learn semantics, or just statistical coincidences?"*

---



---

## 📑 Table of Contents

* [🎓 Socratic Deep Check](#socratic-deep-check)
* [1 · The Supervised Bottleneck & The Self-Supervised Revolution](#1-the-supervised-bottleneck-the-self-supervised-revolution)
  * [The Old Way: Task-Specific Supervised Learning](#the-old-way-task-specific-supervised-learning)
  * [The New Way: Scale via Self-Supervision](#the-new-way-scale-via-self-supervision)
* [2 · The Architectural Trinity](#2-the-architectural-trinity)
  * [A. Encoder-only (Auto-encoding)](#a-encoder-only-auto-encoding)
  * [B. Decoder-only (Auto-regressive)](#b-decoder-only-auto-regressive)
  * [C. Encoder-Decoder (Seq2Seq)](#c-encoder-decoder-seq2seq)
  * [3.1 · The Anatomy of BERT Tensors & The CLS Pooler](#31-the-anatomy-of-bert-tensors-the-cls-pooler)
    * [1. The Trinity of Input Embeddings](#1-the-trinity-of-input-embeddings)
    * [2. The Mechanics of the `[CLS]` Token](#2-the-mechanics-of-the-cls-token)
    * [3. The Pooler Layer: Bridging General to Specific](#3-the-pooler-layer-bridging-general-to-specific)
    * [3.1.1 · The Limitation of Absolute Positional Encoding](#311-the-limitation-of-absolute-positional-encoding)
      * [1. BERT's Absolute Coordinates](#1-berts-absolute-coordinates)
      * [2. The Linguistic Critique](#2-the-linguistic-critique)
      * [3. The Evolutionary Fix: Relative Positional Representations](#3-the-evolutionary-fix-relative-positional-representations)
  * [3.2 · The Semantic Similarity Flaw & Sentence-BERT (SBERT)](#32-the-semantic-similarity-flaw-sentence-bert-sbert)
  * [🎓 Socratic Deep Check](#socratic-deep-check)
  * [The Failure of Raw BERT Embeddings](#the-failure-of-raw-bert-embeddings)
  * [The Solution: Sentence-BERT (SBERT)](#the-solution-sentence-bert-sbert)
    * [1. Siamese Architecture](#1-siamese-architecture)
    * [2. Pooling & Interaction](#2-pooling-interaction)
    * [3. The Objective: Contrastive / Triplet Loss](#3-the-objective-contrastive-triplet-loss)
* [3 · Deep Dive: BERT (Bidirectional Encoder Representations from Transformers)](#3-deep-dive-bert-bidirectional-encoder-representations-from-transformers)
  * [Objective 1: Masked Language Modeling (MLM)](#objective-1-masked-language-modeling-mlm)
  * [Whole Word Masking (WWM)](#whole-word-masking-wwm)
  * [Objective 2: Next Sentence Prediction (NSP)](#objective-2-next-sentence-prediction-nsp)
* [4 · Deep Dive: GPT-style Pre-training](#4-deep-dive-gpt-style-pre-training)
  * [Causal Language Modeling (CLM)](#causal-language-modeling-clm)
* [5 · Post-BERT Optimizations (The Evolution)](#5-post-bert-optimizations-the-evolution)
    * [5.1 · The NSP Failure & ALBERT's Sentence Order Prediction (SOP)](#51-the-nsp-failure-alberts-sentence-order-prediction-sop)
      * [1. Why Next Sentence Prediction (NSP) Failed](#1-why-next-sentence-prediction-nsp-failed)
      * [2. Sentence Order Prediction (SOP): The Harder Challenge](#2-sentence-order-prediction-sop-the-harder-challenge)
* [5b · BERTology and Cross-Lingual Transfer](#5b-bertology-and-cross-lingual-transfer)
  * [BERTology: The Study of Internal Representations](#bertology-the-study-of-internal-representations)
  * [Multilingual Pre-training (mBERT & XLM)](#multilingual-pre-training-mbert-xlm)
* [6 · Implementation: Contextualized Embeddings](#6-implementation-contextualized-embeddings)
* [7 · `AutoModelForMaskedLM` vs `AutoModel` — The LM Head Demystified](#7-automodelformaskedlm-vs-automodel-the-lm-head-demystified)
  * [🎓 The Key Insight Students Miss](#the-key-insight-students-miss)
  * [The Architecture Diagram](#the-architecture-diagram)
  * [When to Use Which?](#when-to-use-which)
* [8 · Hands-On: Pre-training a Mini-BERT from Scratch](#8-hands-on-pre-training-a-mini-bert-from-scratch)
  * [🎓 Junior to Senior: The Concept Bridge](#junior-to-senior-the-concept-bridge)
  * [8.1 · Step 1: Define the Model Architecture with `BertConfig`](#81-step-1-define-the-model-architecture-with-bertconfig)
  * [8.2 · Step 2: Prepare the Toy Training Dataset](#82-step-2-prepare-the-toy-training-dataset)
  * [8.3 · Step 3: The Data Collator — Automatic MLM Masking](#83-step-3-the-data-collator-automatic-mlm-masking)
  * [8.4 · Step 4: Pre-training with the HuggingFace `Trainer`](#84-step-4-pre-training-with-the-huggingface-trainer)
  * [8.5 · Step 5: Test the Pre-trained Mini-BERT](#85-step-5-test-the-pre-trained-mini-bert)
  * [8.6 · Summary: The Pre-training Pipeline](#86-summary-the-pre-training-pipeline)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: The 80/10/10 Rule](#q1-the-801010-rule)
  * [Q2: Architecture Selection](#q2-architecture-selection)
  * [Q3: Causal LM Limitation](#q3-causal-lm-limitation)
  * [Q4: `AutoModel` vs `AutoModelForMaskedLM`](#q4-automodel-vs-automodelformaskedlm)
  * [Q5: Loss Interpretation](#q5-loss-interpretation)
* [🔨 Practical Practice](#practical-practice)


In [ ]:
!pip install -q torch transformers datasets accelerate

## 1 · The Supervised Bottleneck & The Self-Supervised Revolution

### The Old Way: Task-Specific Supervised Learning
Until ~2018, NLP models (LSTMs, early Transformers) were mostly trained from scratch on task-specific labeled data. 
- **Problem:** Labeled data is scarce and expensive ($$$).
- **Consequence:** Models were shallow and struggled with rare words or complex syntax.

### The New Way: Scale via Self-Supervision
The breakthrough was realizing that the **text itself is the label**. By hiding parts of a sentence and asking the model to predict them, we can use the entire internet as a training set.

**The Paradigm Shift:**
1. **Pre-training:** Train a massive model on 100B+ tokens of unlabeled text (General Knowledge).
2. **Fine-tuning:** Adapt that model to a specific task (Sentiment, QA) with a tiny amount of labeled data (Specialization).

| Feature | Static Embeddings (Word2Vec) | Modern Pre-training (BERT/GPT) |
|---------|-----------------------------|-------------------------------|
| **Context** | Context-independent (1 vector per word) | Contextualized (vector depends on neighbors) |
| **Training** | Shallow (Single layer) | Deep (12-96+ layers) |
| **Transfer** | Just the embeddings | Entire model weights |
| **Polysemy** | Struggles with 'bank', 'crane' | Solves by reading context |

---
## 2 · The Architectural Trinity

Transformers are like LEGO blocks; how you stack them determines what they learn.

### A. Encoder-only (Auto-encoding)
- **Examples:** BERT, RoBERTa, ALBERT.
- **Attention:** Bidirectional (sees left AND right context).
- **Best For:** Natural Language Understanding (NLU) — Classification, NER, Sentiment.

### B. Decoder-only (Auto-regressive)
- **Examples:** GPT series, Llama, Mistral.
- **Attention:** Causal (masked) — only sees past tokens.
- **Best For:** Natural Language Generation (NLG) — Chatbots, Creative writing.

### C. Encoder-Decoder (Seq2Seq)
- **Examples:** T5, BART.
- **Attention:** Bidirectional encoder + Causal decoder.
- **Best For:** Conditional Generation — Translation, Summarization.

---

### 3.1 · The Anatomy of BERT Tensors & The CLS Pooler

Before BERT processes a single word, it transforms raw tokens into additive embeddings. Understanding this tensor layout is critical for engineering custom classification heads.

#### 1. The Trinity of Input Embeddings
BERT doesn't just look at word IDs. Each input is a sum of three vectors:
- **Token Embeddings**: Standard WordPiece IDs.
- **Position Embeddings**: Learned vectors representing absolute position (0...511).
- **Segment Embeddings (`token_type_ids`)**: Used for sentence-pair tasks (e.g., NSP).

**Visual Mapping (Two-Sentence Input):**
Sentence A: "Hello world" | Sentence B: "How are you?"
```text
Tokens:   [CLS]  Hello  world  [SEP]   How    are    you?   [SEP]
Token IDs: 101    7592   2088   102     2129   2024   2017   102
Segments:  0      0      0      0       1      1      1      1    <-- token_type_ids
```
**Total Embedding = Token(101) + Pos(0) + Seg(0)**

#### 2. The Mechanics of the `[CLS]` Token
In a Transformer, every token's output depends on every other token via self-attention. By the time data reaches the final layer, the first token (`[CLS]`) has "attended" to the entire sequence multiple times.
- **Engineering Design**: Instead of averaging all tokens, we designate `[CLS]` as the sequence summary token. Its vector becomes a high-dimensional bottleneck representing the distilled meaning of the entire input.

#### 3. The Pooler Layer: Bridging General to Specific
If you extract base BERT output using `AutoModel`, you get a sequence of hidden tokens. However, the `BertForSequenceClassification` class performs an extra step called the **Pooler**.
- **The Operation**: `[CLS] Hidden State` → `Linear(768, 768)` → `Tanh()` → `Classification Head`.
- **The Rationale**: The raw `[CLS]` state is optimized for MLM (reconstructing words). The Pooler layer acts as an additional learnable transformation that helps "buffer" this general representation, making it easier for the model to adapt to downstream tasks (like sentiment) during fine-tuning.


#### 3.1.1 · The Limitation of Absolute Positional Encoding

##### 1. BERT's Absolute Coordinates
BERT (and the original Transformer) uses **Absolute Positional Encodings**. It learns a unique vector for index 0, index 1, index 2, up to 511. This means the model knows *where* a word is in a sentence relative to the very first token.

##### 2. The Linguistic Critique
From a linguistic and biological perspective, this is counter-intuitive.
- **The Argument**: The meaning of a word depends on its **distance** to its neighbors, not its absolute index in a document.
- **Example**: In the sentences *"The dog chased the cat"* and *"Yesterday afternoon, the dog chased the cat"*, the relationship between *"dog"* and *"chased"* is identical. However, in BERT, the indices of these words shift from [1, 2] to [3, 4], forcing the model to re-learn the same dependency at every possible absolute position.

##### 3. The Evolutionary Fix: Relative Positional Representations
Modern architectures like **DeBERTa** and **T5** use **Relative Positional Encoding**. Instead of adding a vector to the embedding at the start, they modify the **Attention Mechanism** itself.
- When computing the attention score between word $i$ and word $j$, the model injects a bias based on $|i - j|$.
- This ensures the model learns that a subject-verb relationship "looks" the same whether it happens at the start of a sentence or the end of a 500-token document.


---
### 3.2 · The Semantic Similarity Flaw & Sentence-BERT (SBERT)

### 🎓 Socratic Deep Check
> *"If BERT is so good at understanding context, why does it fail at the simplest semantic task? If you take two sentences—'The man is playing guitar' and 'A person is strumming a musical instrument'—and compare their raw [CLS] vectors using Cosine Similarity, the score is often lower than comparing 'The man is playing guitar' with 'The man is eating an apple'. Why does BERT's 'intelligence' fail to translate into geometric distance?"*

### The Failure of Raw BERT Embeddings
The technical reason is hidden in the **Pre-training Objective**. BERT was trained on:
1. **MLM**: Predicting missing words.
2. **NSP**: Predicting if Sentence B follows Sentence A.

Neither of these objectives forces the model to ensure that semantically similar sentences occupy the same neighborhood in vector space. Consequently, raw BERT embeddings suffer from **anisotropy** (occupying a narrow cone in vector space) and lack a "metric" property. Using raw `[CLS]` vectors for clustering or semantic search yields results often worse than simple GloVe averages.

### The Solution: Sentence-BERT (SBERT)
Introduced by **Reimers & Gurevych (2019)**, SBERT uses a **Siamese Network** architecture to "fine-tune" BERT specifically for semantic similarity.

#### 1. Siamese Architecture
We pass Sentence A and Sentence B through two **identical** BERT models (sharing the exact same weights). 

#### 2. Pooling & Interaction
Instead of just using `[CLS]`, SBERT typically uses **MEAN Pooling** (averaging all output token vectors). The model then computes a representation that captures the interaction between the two sentences (e.g., concatenating $|u|, |v|,$ and $|u-v|$).

#### 3. The Objective: Contrastive / Triplet Loss
During training, SBERT uses a **Contrastive Loss** or **Triplet Loss**:
- **Positive Pairs**: "Similar sentences" are pulled together (Cosine Similarity $\to 1$).
- **Negative Pairs**: "Dissimilar sentences" are pushed apart (Cosine Similarity $\to 0$).

**The Result:** BERT is transformed from a word-prediction engine into a powerful **Embedding Engine**, capable of performing semantic search across millions of documents in milliseconds.

> **🎓 Masterclass Insight:** This transition from NLU (Natural Language Understanding) to Semantic Space is what makes **Vector Databases** and **RAG (Retrieval-Augmented Generation)** possible. Without SBERT-style training, your vector search would be no better than a random guess.


## 3 · Deep Dive: BERT (Bidirectional Encoder Representations from Transformers)

BERT was the first model to successfully use a deep bidirectional encoder for pre-training.

### Objective 1: Masked Language Modeling (MLM)
BERT hides 15% of tokens. To prevent the model from only learning to see the `[MASK]` token, it uses the **80/10/10 Rule**:
- **80%** of the time: Replace with `[MASK]`.
- **10%** of the time: Replace with a **random word** (forces the model to check if the word fits).
- **10%** of the time: Keep it **unchanged** (forces the model to always build good representations).

### Whole Word Masking (WWM)
**The Vulnerability of Standard MLM:** Standard MLM masks individual *subword tokens*. If the word "unbelievable" is tokenized into `["un", "##believ", "##able"]` and only `"##believ"` is randomly masked, the model sees `["un", "[MASK]", "##able"]`. This leaks massive hints through the prefixes and suffixes, making the pre-training task far too easy; the model can guess the subword without properly learning the complex sentence context.

**The Solution:** Whole Word Masking (WWM) forces the algorithm to mask *all* subwords corresponding to a single word at once.

**Illustration:**
- **Tokens:** `["un", "##believ", "##able"]`
- **Standard MLM:** `["un", "[MASK]", "##able"]` *(Too easy)*
- **WWM:** `["[MASK]", "[MASK]", "[MASK]"]` *(Forces deeper contextual learning)*

### Objective 2: Next Sentence Prediction (NSP)
Model receives two segments (A and B). It must predict if B follows A. 
- *Note: Later models like RoBERTa found that removing NSP actually improved performance on most downstream tasks.*

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModelForMaskedLM.from_pretrained('bert-base-uncased')

def probe_bert(text: str) -> None:
    """Probe BERT's knowledge by filling a [MASK] token.
    
    This demonstrates that BERT has learned factual and linguistic
    knowledge purely from the MLM pre-training objective.
    """
    inputs = tokenizer(text, return_tensors='pt')
    mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
    
    with torch.no_grad():
        logits = model(**inputs).logits
        
    mask_token_logits = logits[0, mask_token_index, :]
    top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()
    
    print(f"Text: {text}")
    for i, token in enumerate(top_5_tokens):
        print(f"  Top {i+1}: {tokenizer.decode([token])}")

probe_bert("The capital of France is [MASK].")
probe_bert("The chef prepared a delicious [MASK] for the guests.")

---
## 4 · Deep Dive: GPT-style Pre-training

While BERT looks at context bidirectionally, GPT looks **Forward Only**.

### Causal Language Modeling (CLM)
The objective is simple: Predict token $t$ given $t-1, t-2, ... 1$. 

**The Mathematical View:**
Maximize the likelihood: $L = \sum_i \log P(x_i | x_{<i}; \theta)$

**Why it's better for generation:**
Since the model never sees the "future" during training, it doesn't cheat. This makes it a natural fit for sampling one word at a time to generate text.

---
## 5 · Post-BERT Optimizations (The Evolution)

After BERT, several models refined the recipe:

1. **RoBERTa (The Robust BERT):**
   - Removed NSP (found it unnecessary).
   - Dynamic Masking: Masking patterns change every epoch (BERT mask was static).
   - 10x more data, 10x longer training.

2. **ALBERT (A Lite BERT):**
   - **Factorized Embedding:** Separate vocabulary size from hidden layer size.
   - **Cross-Layer Parameter Sharing:** All 12 layers share the SAME weights. 90% fewer parameters with similar performance!

3. **ELECTRA (Replaced Token Detection):**
   - Instead of hiding tokens, it replaces some with plausibly wrong words from a small "generator" network.
   - The model acts as a **Discriminator** (Is this word original or fake?).
   - Much more efficient because it learns from EVERY token, not just the 15% masked ones.

#### 5.1 · The NSP Failure & ALBERT's Sentence Order Prediction (SOP)

##### 1. Why Next Sentence Prediction (NSP) Failed
The original BERT was trained with **NSP**: given Sentences A and B, predict if B follows A. While logically sound, it proved to be a "broken" objective in practice.
- **Topic-Shift Detection**: NSP ended up being too easy. If Sentence A is about *Quantum Physics* and Sentence B is about *Ice Cream*, the model can predict "No" just by looking at the vocabulary mismatch (topic shift), without ever learning the actual relationship between sentences.

##### 2. Sentence Order Prediction (SOP): The Harder Challenge
**ALBERT** (Lan et al., 2019) replaced NSP with **Sentence Order Prediction (SOP)**.
- **The Task**: The model is always given two **consecutively** following sentences from the same document.
- **The Twist**: In 50% of cases, the order is **correct** (A $\rightarrow$ B). In the other 50%, the order is **swapped** (B $\rightarrow$ A).
- **The Goal**: To solve SOP, the model cannot rely on topic shift (since both sentences are from the same context). It *must* understand **coherence, discourse flow, and temporal order**. This forces the model to learn much deeper linguistic representations.


---
## 5b · BERTology and Cross-Lingual Transfer

### BERTology: The Study of Internal Representations
As BERT became the foundation of NLP, a new field of empirical study emerged: **BERTology**. Researchers began "probing" BERT's layers to understand what exactly happens inside its 12-to-24 Transformer blocks.

**The Hierarchical Discovery:**
- **Lower Layers (1-4)**: Primarily learn **Syntax**. They capture part-of-speech tags, dependency relations, and basic grammar rules.
- **Middle Layers (5-8)**: Transition toward **Coreference and Structure**.
- **Higher Layers (9-12)**: Primarily learn **Semantics**. They capture task-specific meaning, disambiguate polysemous words, and handle complex reasoning.

### Multilingual Pre-training (mBERT & XLM)
One of the most profound "accidental" discoveries in NLP was **mBERT**. Researchers trained a standard BERT model on the concatenated Wikipedia datasets of 100+ different languages (English, French, Hindi, Chinese, etc.) using the *exact same* Masked Language Modeling objective.

**The Result: A Shared Latent Space**
Even though the model was never given explicit translation pairs (e.g., "Apple" = "Pomme"), it "magically" aligned the languages in its internal vector space. 
- The model learned that the *contextual role* of "Apple" in English sentences is identical to the role of "Pomme" in French.
- **Zero-Shot Cross-Lingual Transfer**: This allows a practitioner to fine-tune a model on English sentiment data and immediately deploy it on French, Spanish, or German text with high accuracy — without ever seeing a single labeled French sentence!


---
## 6 · Implementation: Contextualized Embeddings

Let's prove that BERT word vectors change based on context. We will compare the vector for the word **"bank"** across three sentences.

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
from torch.nn.functional import cosine_similarity

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased')

sentences = [
    "I need to go to the bank to deposit my paycheck.",
    "The river bank was muddy after the rain.",
    "You can always bank on her to get the job done."
]

embeddings = []

for sent in sentences:
    inputs = tokenizer(sent, return_tensors='pt', padding=True, truncation=True)
    # Note: bert-base-uncased tokenizes these 'bank's as a single token [2910]
    # We find the position in the current sentence input_ids
    token_ids = inputs['input_ids'][0].tolist()
    token_idx = token_ids.index(2910)
    
    with torch.no_grad():
        outputs = model(**inputs)
        # Get the last hidden state for the 'bank' token
        # Shape: [batch, seq_len, hidden_dim]
        emb = outputs.last_hidden_state[0, token_idx, :]
        embeddings.append(emb)

print("Cosine Similarity Matrix for 'bank':")
print(f"Financial vs River:    {cosine_similarity(embeddings[0].unsqueeze(0), embeddings[1].unsqueeze(0)).item():.4f}")
print(f"Financial vs Verb:     {cosine_similarity(embeddings[0].unsqueeze(0), embeddings[2].unsqueeze(0)).item():.4f}")
print(f"River vs Verb:         {cosine_similarity(embeddings[1].unsqueeze(0), embeddings[2].unsqueeze(0)).item():.4f}")

print("\nObservation: Even though the token ID is identical (2910), the vectors are distinct!")
print("This is the core power of contextualized embeddings.")

---
## 7 · `AutoModelForMaskedLM` vs `AutoModel` — The LM Head Demystified

### 🎓 The Key Insight Students Miss

When browsing HuggingFace code, you'll encounter two seemingly similar classes:

```python
model_a = AutoModel.from_pretrained('bert-base-uncased')           # "Base" model
model_b = AutoModelForMaskedLM.from_pretrained('bert-base-uncased') # "LM Head" model
```

These are **NOT the same model**. Understanding the difference is crucial to understanding
how pre-training works and how to transfer knowledge to downstream tasks.

### The Architecture Diagram

```
                    AutoModel                         AutoModelForMaskedLM
              ┌──────────────────┐              ┌──────────────────────────────┐
              │                  │              │                              │
  Input IDs ──▶  Embeddings     │  Input IDs ──▶  Embeddings                  │
              │       │         │              │       │                       │
              │  12x Transformer│              │  12x Transformer Layers       │
              │     Layers      │              │       │                       │
              │       │         │              │  ┌────▼────────────────────┐  │
              │       ▼         │              │  │   LM Head               │  │
              │  Hidden States  │              │  │  Dense(768→768) + GELU  │  │
              │   (768-dim)     │              │  │  LayerNorm              │  │
              │                 │              │  │  Linear(768→30522)      │  │
              └──────────────────┘              │  │  = logits over VOCAB   │  │
                                               │  └────────────────────────┘  │
                    OUTPUT:                    └──────────────────────────────┘
              768-dim vector per token
              (general representation)                 OUTPUT:
                                               30522-dim vector per token
                                               (probability of each vocab word)
```

### When to Use Which?

| Use Case | Class | Why |
|----------|-------|-----|
| **Pre-training** (MLM) | `AutoModelForMaskedLM` | Needs the LM head to predict masked tokens |
| **Fine-tuning** (Classification) | `AutoModel` + custom head | Replace LM head with your own classifier |
| **Sentence Embeddings** | `AutoModel` | Extract [CLS] or mean-pool hidden states |
| **Feature Extraction** | `AutoModel` | Use hidden states as features for other models |

> **🧠 Socratic Question:** *During pre-training, the LM head maps 768 dimensions to 30,522*
> *(vocabulary size). After pre-training, we throw this head away for fine-tuning. If it's*
> *discarded, why not skip it entirely? What does the LM head force the backbone to learn*
> *that a simpler objective wouldn't?*

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Hands-on: Inspect the architectural difference
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# ── Load BOTH model variants ──────────────────────────────────────────────────
base_model = AutoModel.from_pretrained('bert-base-uncased')
mlm_model  = AutoModelForMaskedLM.from_pretrained('bert-base-uncased')

# ── Count parameters ─────────────────────────────────────────────────────────
base_params = sum(p.numel() for p in base_model.parameters())
mlm_params  = sum(p.numel() for p in mlm_model.parameters())
head_params = mlm_params - base_params

print('=' * 65)
print('PARAMETER COMPARISON: AutoModel vs AutoModelForMaskedLM')
print('=' * 65)
print(f'  AutoModel (base):           {base_params:>12,} parameters')
print(f'  AutoModelForMaskedLM:       {mlm_params:>12,} parameters')
print(f'  LM Head overhead:           {head_params:>12,} parameters ({head_params/base_params*100:.1f}%)')
print()

# ── Compare output shapes ────────────────────────────────────────────────────
text = "The [MASK] sat on the mat."
inputs = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    base_out = base_model(**inputs)
    mlm_out  = mlm_model(**inputs)

print('OUTPUT SHAPE COMPARISON:')
print(f'  AutoModel output:           {base_out.last_hidden_state.shape}')
print(f'    → {base_out.last_hidden_state.shape[-1]}-dim embedding per token (general representation)')
print()
print(f'  AutoModelForMaskedLM output: {mlm_out.logits.shape}')
print(f'    → {mlm_out.logits.shape[-1]}-dim logits per token (one score per vocab word)')
print()

# ── Show what the LM head actually is ────────────────────────────────────────
print('LM HEAD ARCHITECTURE (the extra layers):')
print('─' * 65)
# The LM head in BERT is under mlm_model.cls
if hasattr(mlm_model, 'cls'):
    print(mlm_model.cls)
else:
    # Some model variants store it differently
    print(mlm_model)
print('─' * 65)
print()
print('Key insight: The LM Head is Dense(768→768) + GELU + LayerNorm + Linear(768→vocab).')
print('This head is ONLY needed during pre-training. For fine-tuning, you swap it out.')

---
## 8 · Hands-On: Pre-training a Mini-BERT from Scratch

### 🎓 Junior to Senior: The Concept Bridge

**The Senior Perspective:** Juniors use `from_pretrained()` and treat the model as a black box.
Seniors understand *exactly* what happened during pre-training because they've done it themselves.
In this section, we will initialize a **randomly-weighted miniature BERT** and train it on a 
toy corpus using the exact same MLM objective that Google used to create the original BERT.

**What you will learn:**
1. How to configure a BERT from scratch using `BertConfig`
2. How `DataCollatorForLanguageModeling` implements the 80/10/10 masking strategy
3. How the HuggingFace `Trainer` API abstracts the training loop
4. Why pre-training is just "predicting masked words" at scale

**⚠️ Important:** We use a *tiny* model and *tiny* dataset for pedagogical purposes. Real 
BERT pre-training uses 16 TPUs for 4 days on 3.3B words. Our mini-BERT will not produce 
useful language understanding — but it *will* demonstrate that the loss decreases, proving 
the model is learning the structure of its training data.

---

### 8.1 · Step 1: Define the Model Architecture with `BertConfig`

In [ ]:
from transformers import BertConfig, BertForMaskedLM, BertTokenizerFast

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 1: Define a miniature BERT configuration
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# Compare with the real BERT-base:
#   BERT-base: hidden=768,  layers=12, heads=12, params=110M
#   Mini-BERT: hidden=128,  layers=2,  heads=2,  params=~600K
#
# This is ~180x smaller. Perfect for learning on a laptop.

# We'll reuse the existing bert-base-uncased tokenizer (vocab_size=30522)
# In production, you'd train your own tokenizer on your corpus first.
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

config = BertConfig(
    vocab_size=tokenizer.vocab_size,  # 30522 — same vocabulary as BERT-base
    hidden_size=128,                  # Embedding dimension (BERT-base: 768)
    num_hidden_layers=2,              # Transformer layers (BERT-base: 12)
    num_attention_heads=2,            # Attention heads (BERT-base: 12)
    intermediate_size=512,            # FFN inner dimension (BERT-base: 3072)
    max_position_embeddings=128,      # Max sequence length (BERT-base: 512)
)

# ── Initialize model with RANDOM weights ─────────────────────────────────────
# This is the key difference from `from_pretrained()`:
#   from_pretrained() → loads weights from a checkpoint (transfer learning)
#   BertForMaskedLM(config) → random Xavier/Kaiming initialization (tabula rasa)
mini_bert = BertForMaskedLM(config)

num_params = sum(p.numel() for p in mini_bert.parameters())
trainable  = sum(p.numel() for p in mini_bert.parameters() if p.requires_grad)

print('=' * 65)
print('MINI-BERT ARCHITECTURE')
print('=' * 65)
print(f'  Total parameters:     {num_params:>10,}')
print(f'  Trainable parameters: {trainable:>10,}')
print(f'  Hidden size:          {config.hidden_size}')
print(f'  Transformer layers:   {config.num_hidden_layers}')
print(f'  Attention heads:      {config.num_attention_heads}')
print(f'  FFN inner dim:        {config.intermediate_size}')
print(f'  Max sequence length:  {config.max_position_embeddings}')
print(f'  Vocabulary size:      {config.vocab_size}')
print()

# ── Verify it's untrained: random predictions should be garbage ──────────────
test_input = tokenizer("The [MASK] is a beautiful day.", return_tensors='pt')
import torch
with torch.no_grad():
    logits = mini_bert(**test_input).logits
    mask_idx = torch.where(test_input['input_ids'] == tokenizer.mask_token_id)[1]
    top_tokens = torch.topk(logits[0, mask_idx, :], 5, dim=1).indices[0].tolist()

print('Random model predictions for "The [MASK] is a beautiful day.":')
for i, tok in enumerate(top_tokens):
    print(f'  Top {i+1}: {tokenizer.decode([tok])}  (random nonsense — model is untrained!)')

### 8.2 · Step 2: Prepare the Toy Training Dataset

We'll create a small in-memory dataset of English sentences. In production,
you'd use massive corpora like Wikipedia + BookCorpus (BERT), The Pile (GPT-NeoX),
or RedPajama (Llama).

**Key Design Decisions:**
- We tokenize all sentences and set a fixed `max_length` for batching efficiency.
- We use the existing `bert-base-uncased` tokenizer. In real pre-training, you'd
  train a tokenizer on your specific corpus first (see NLP 01: BPE/WordPiece).

In [ ]:
from datasets import Dataset

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 2: Create a toy text corpus
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# A diverse set of simple sentences covering different topics.
# The model will learn to predict masked words within these patterns.

toy_corpus = [
    # Science
    "The sun is the closest star to the earth and provides light and heat.",
    "Water freezes at zero degrees celsius and boils at one hundred degrees.",
    "Photosynthesis is the process by which plants convert sunlight into energy.",
    "The human brain contains approximately one hundred billion neurons.",
    "Gravity is the force that attracts objects toward the center of the earth.",
    "DNA contains the genetic instructions for the development of living organisms.",
    "The speed of light is approximately three hundred thousand kilometers per second.",
    "Atoms are the basic building blocks of all matter in the universe.",
    
    # Technology
    "Machine learning models learn patterns from data without explicit programming.",
    "Neural networks are inspired by the structure of the human brain.",
    "Deep learning uses multiple layers to learn hierarchical representations.",
    "Natural language processing enables computers to understand human language.",
    "The transformer architecture revolutionized the field of artificial intelligence.",
    "Gradient descent is used to optimize the parameters of a neural network.",
    "Transfer learning allows models to reuse knowledge from one task to another.",
    "Attention mechanisms allow models to focus on the most relevant parts of the input.",
    
    # General knowledge
    "The capital of France is Paris and it is known for the Eiffel Tower.",
    "Shakespeare wrote many famous plays including Romeo and Juliet.",
    "The Amazon rainforest is the largest tropical forest in the world.",
    "Music is a universal language that connects people across cultures.",
    "The Pacific Ocean is the largest and deepest ocean on the planet.",
    "Leonardo da Vinci painted the Mona Lisa in the early sixteenth century.",
    "The Great Wall of China was built over many centuries to protect the border.",
    "Democracy is a system of government where citizens vote for their leaders.",
    
    # Repeated patterns (helps the tiny model find regularities)
    "The cat sat on the mat and looked out the window.",
    "The dog ran through the park and played with the ball.",
    "The bird flew over the tall trees and into the blue sky.",
    "The fish swam in the clear water of the mountain lake.",
    "The teacher explained the lesson to the students in the classroom.",
    "The doctor examined the patient and prescribed the right medicine.",
    "The engineer designed the bridge to withstand strong winds and earthquakes.",
    "The scientist conducted the experiment and recorded the results carefully.",
]

# ── Tokenize the corpus ──────────────────────────────────────────────────────
def tokenize_function(examples: dict) -> dict:
    """Tokenize texts with truncation and padding to a fixed length.

    Returns input_ids, attention_mask, and token_type_ids ready for MLM.
    """
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=64,        # Short sequences for our toy dataset
        return_special_tokens_mask=True,  # Needed by DataCollator to avoid masking [CLS]/[SEP]
    )

# Create HuggingFace Dataset and tokenize
raw_dataset = Dataset.from_dict({'text': toy_corpus})
tokenized_dataset = raw_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],  # We only need token IDs now
    desc='Tokenizing corpus',
)

print(f'Dataset size: {len(tokenized_dataset)} samples')
print(f'Sample features: {list(tokenized_dataset.features.keys())}')
print(f'Sample input_ids shape: {len(tokenized_dataset[0]["input_ids"])} tokens')
print(f'\nFirst sentence (decoded):')
print(f'  {tokenizer.decode(tokenized_dataset[0]["input_ids"], skip_special_tokens=True)}')

### 8.3 · Step 3: The Data Collator — Automatic MLM Masking

The `DataCollatorForLanguageModeling` is where the magic happens. At each training step,
it **dynamically** applies the 80/10/10 masking strategy to the batch:

```
Original:   [CLS] the cat sat on the mat [SEP] [PAD] [PAD]
                          ↓ DataCollator (mlm_probability=0.15)
input_ids:  [CLS] the cat [MASK] on the mat [SEP] [PAD] [PAD]  ← model input
labels:     [-100 -100 -100  sat -100 -100 -100 -100 -100 -100] ← loss target
```

**Key Details:**
- Labels are `-100` for non-masked positions → PyTorch's `CrossEntropyLoss` ignores them.
- Special tokens (`[CLS]`, `[SEP]`, `[PAD]`) are NEVER masked.
- Masking is **dynamic**: different random masks each epoch (like RoBERTa, unlike original BERT).

In [ ]:
from transformers import DataCollatorForLanguageModeling

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 3: Create the MLM Data Collator
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,                # Masked Language Modeling (set False for CLM/GPT)
    mlm_probability=0.15,    # Mask 15% of tokens (same as BERT paper)
)

# ── Demonstrate what the collator does ────────────────────────────────────────
# Take a small batch and see the masking in action
sample_batch = [tokenized_dataset[i] for i in range(4)]
collated = data_collator(sample_batch)

print('DataCollatorForLanguageModeling — Masking Demo')
print('=' * 70)
print(f'Batch input_ids shape: {collated["input_ids"].shape}')
print(f'Batch labels shape:    {collated["labels"].shape}')
print()

# Show the first sample's masking
input_ids = collated['input_ids'][0]
labels = collated['labels'][0]

print('Sample 0 — Masking Visualization:')
print('─' * 70)

masked_count = 0
total_real_tokens = 0

for i, (inp, lab) in enumerate(zip(input_ids, labels)):
    inp_tok = tokenizer.decode([inp.item()]).strip()
    if inp.item() in (tokenizer.pad_token_id,):
        continue  # Skip padding for clarity
    total_real_tokens += 1
    if lab.item() != -100:
        original = tokenizer.decode([lab.item()]).strip()
        masked_count += 1
        print(f'  Position {i:2d}: {inp_tok:12s} ← label: "{original}" (MASKED — model must predict this)')
    else:
        print(f'  Position {i:2d}: {inp_tok:12s}    (label: -100, ignored by loss)')

print(f'\nMasked {masked_count}/{total_real_tokens} tokens '
      f'({masked_count/max(total_real_tokens,1)*100:.1f}% — target is 15%)')

### 8.4 · Step 4: Pre-training with the HuggingFace `Trainer`

The `Trainer` API wraps the entire training loop:
- Forward pass → compute loss (cross-entropy on masked positions)
- Backward pass → compute gradients
- Optimizer step → update weights (AdamW by default)
- Logging → track loss, learning rate, throughput

**What to expect:**
- Initial loss ≈ `ln(30522) ≈ 10.3` (random guessing over the full vocabulary)
- Loss should decrease significantly over a few epochs
- Final loss will NOT reach 0 — the model can't perfectly memorize with only 2 layers

In [ ]:
import math
from transformers import TrainingArguments, Trainer

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 4: Configure and run the Trainer
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Theoretical maximum loss: random guessing over entire vocabulary
theoretical_max_loss = math.log(tokenizer.vocab_size)
print(f'Theoretical initial loss (random guessing): ln({tokenizer.vocab_size}) = {theoretical_max_loss:.2f}')
print()

training_args = TrainingArguments(
    output_dir='./mini_bert_pretrain',
    
    # Training hyperparameters
    num_train_epochs=30,              # More epochs to compensate for tiny dataset
    per_device_train_batch_size=8,    # Small batch for our 32 samples
    learning_rate=5e-4,               # Higher LR for small model (BERT used 1e-4)
    weight_decay=0.01,                # L2 regularization (same as BERT)
    warmup_ratio=0.1,                 # Linear warmup for 10% of training steps
    
    # Optimizer
    optim='adamw_torch',              # AdamW (decoupled weight decay)
    
    # Logging
    logging_strategy='epoch',         # Log loss at each epoch
    save_strategy='no',               # Don't save checkpoints (toy experiment)
    report_to='none',                 # Disable W&B / MLflow for this demo
    
    # Performance
    fp16=False,                       # Keep FP32 for stability on CPU
    dataloader_num_workers=0,         # Avoid multiprocessing overhead on small data
    seed=42,                          # Reproducibility
)

trainer = Trainer(
    model=mini_bert,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print('Starting Mini-BERT pre-training...')
print('Watch the loss decrease from ~10.3 (random) towards lower values.')
print('─' * 65)

# ── Train! ────────────────────────────────────────────────────────────────────
train_result = trainer.train()

print('─' * 65)
print(f'Training complete!')
print(f'  Final training loss: {train_result.training_loss:.4f}')
print(f'  Initial random loss: ~{theoretical_max_loss:.2f}')
print(f'  Loss reduction:      {theoretical_max_loss - train_result.training_loss:.2f} nats')
print(f'  Perplexity:          {math.exp(train_result.training_loss):.1f}')
print(f'    (Random baseline:  {tokenizer.vocab_size} | Perfect: 1.0)')

### 8.5 · Step 5: Test the Pre-trained Mini-BERT

Let's see if our tiny model learned anything from its training corpus.
Since we only trained on 32 sentences, the model will only "know" patterns
from that specific data. This is the **microscopic version** of what happens
when Google trains BERT on all of Wikipedia.

In [ ]:
import torch

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 5: Evaluate the trained Mini-BERT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

mini_bert.eval()

def probe_mini_bert(text: str, top_k: int = 5) -> None:
    """Probe the trained mini-BERT to fill a [MASK] token.
    
    Args:
        text: Input text containing one [MASK] token.
        top_k: Number of top predictions to show.
    """
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=64)
    mask_idx = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)[1]
    
    if len(mask_idx) == 0:
        print(f'  No [MASK] token found in: "{text}"')
        return
    
    with torch.no_grad():
        logits = mini_bert(**inputs).logits
    
    mask_logits = logits[0, mask_idx[0], :]
    probs = torch.softmax(mask_logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, top_k)
    
    print(f'  Text: "{text}"')
    for i, (prob, idx) in enumerate(zip(top_probs, top_indices)):
        word = tokenizer.decode([idx.item()]).strip()
        print(f'    Top {i+1}: "{word}" (confidence: {prob.item():.4f})')
    print()


print('=' * 65)
print('MINI-BERT PREDICTIONS (after pre-training on toy corpus)')
print('=' * 65)
print()

# ── Test on patterns FROM the training data ──────────────────────────────────
print('▸ In-distribution tests (patterns seen during training):')
print('─' * 65)
probe_mini_bert("The cat sat on the [MASK] and looked out the window.")
probe_mini_bert("The capital of France is [MASK].")
probe_mini_bert("Machine learning [MASK] learn patterns from data.")

# ── Test on patterns NOT in the training data ────────────────────────────────
print('▸ Out-of-distribution tests (patterns NOT seen during training):')
print('─' * 65)
probe_mini_bert("The president of the United States lives in the [MASK].")
probe_mini_bert("Python is a popular [MASK] language.")

print('Key Observations:')
print('  1. In-distribution: The model should show reasonable predictions for patterns it trained on.')
print('  2. Out-of-distribution: Predictions will be poor — the model only knows its 32 sentences!')
print('  3. This is WHY real pre-training needs billions of tokens — more data = more general knowledge.')
print()
print('  💡 Scaling Insight: Increasing data from 32 sentences → 3.3B words (BERT) → 15T tokens (Llama 3)')
print('     is what transforms a toy model into a powerful language understanding system.')

### 8.6 · Summary: The Pre-training Pipeline

We just executed the **exact same pipeline** that Google used to create BERT:

```
┌────────────────┐    ┌──────────────────┐    ┌────────────────────┐    ┌────────────────┐
│  1. BertConfig │───▶│ 2. Tokenize Text │───▶│ 3. DataCollator    │───▶│ 4. Trainer     │
│  (Architecture)│    │ (IDs + Padding)  │    │ (Dynamic Masking)  │    │ (Optimization) │
└────────────────┘    └──────────────────┘    └────────────────────┘    └────────────────┘
  hidden_size=128       max_length=64           mlm_probability=0.15    epochs=30
  layers=2              tokenizer=BERT          80/10/10 strategy       lr=5e-4
  heads=2                                                               AdamW
```

**The only differences in production:**

| Dimension | Our Mini-BERT | Google's BERT-base |
|-----------|--------------|--------------------|
| Parameters | ~600K | 110M |
| Training data | 32 sentences | 3.3B words (Wikipedia + BookCorpus) |
| Training compute | ~30 seconds (CPU) | 4 days (16 Cloud TPUs) |
| Vocabulary | Borrowed (bert-base) | Custom WordPiece trained on corpus |
| Result | Memorizes toy patterns | General language understanding |

> **🧠 The Deep Insight:** Pre-training is conceptually simple — it's just "fill in the blank"
> at massive scale. The magic is not in the algorithm (MLM is straightforward) but in the
> **emergent representations** that arise when you scale data and compute. The backbone learns
> syntax, semantics, world knowledge, and reasoning — all as a byproduct of predicting
> missing words. This is why the pre-train → fine-tune paradigm dominates modern NLP.

---

---
## ✅ Knowledge Check

### Q1: The 80/10/10 Rule
**Why does BERT use the 80/10/10 rule instead of always replacing with `[MASK]`?**

<details><summary>👉 Answer</summary>

Because at fine-tuning time, the model never sees `[MASK]`. If it only learned from `[MASK]`, its performance would decouple from real-world text. The 10% random replacement forces the model to maintain robust representations even when the token doesn't look corrupted, and the 10% unchanged forces it to always build contextual representations regardless of what it sees.
</details>

### Q2: Architecture Selection
**Which architecture is best for summarization?**

<details><summary>👉 Answer</summary>

Encoder-Decoder (e.g., T5, BART). You need the encoder to understand the long text and the decoder to generate the short summary. Pure encoder (BERT) can't generate; pure decoder (GPT) can't see bidirectional context for the source.
</details>

### Q3: Causal LM Limitation
**What is the main limitation of Causal LLMs (GPT)?**

<details><summary>👉 Answer</summary>

They cannot see the "future context". For tasks like NER or translation, seeing the whole sentence at once is often better. BERT's bidirectional attention outperforms GPT-style models on understanding tasks precisely because it can attend to both left and right context simultaneously.
</details>

### Q4: `AutoModel` vs `AutoModelForMaskedLM`
**You want to fine-tune BERT for sentiment classification. Should you load `AutoModel` or `AutoModelForMaskedLM`? Why?**

<details><summary>👉 Answer</summary>

`AutoModel`. The MLM head (which maps to vocabulary) is useless for classification — you need hidden states to feed into your own classification head. Loading `AutoModelForMaskedLM` wastes ~24M parameters and gives you logits over 30,522 tokens when you only need 2-5 class probabilities. Alternatively, use `AutoModelForSequenceClassification` which replaces the LM head with a classification head automatically.
</details>

### Q5: Loss Interpretation
**Our mini-BERT started with loss ≈ 10.3 and ended around 4-6. What does a loss of 10.3 correspond to mathematically?**

<details><summary>👉 Answer</summary>

`ln(30522) ≈ 10.33`. This is the cross-entropy loss of a uniform distribution over the entire vocabulary — the model is randomly guessing among 30,522 tokens. A loss of 6.0 means `e^6 ≈ 403` effective choices — the model has narrowed from 30K candidates to ~400. A loss of 4.0 means `e^4 ≈ 55` effective choices. Real BERT achieves loss ~1.5-2.0 on its training data.
</details>

---

## 🔨 Practical Practice

1. **Probing Knowledge:** Use the `AutoModelForMaskedLM` code to see if BERT knows facts like "Barack Obama was the [MASK] of the USA."
2. **Similarity Search:** Take a list of 5 sentences and find which two are most similar using the `[CLS]` token embedding from BERT.
3. **Scale Experiment:** Increase the toy corpus to 100+ sentences and re-run the mini-BERT training. Does the final loss change? Does generalization improve?
4. **Architecture Ablation:** Modify `BertConfig` to use `num_hidden_layers=4` and `hidden_size=256`. How does training loss compare? How many parameters does the model have now?

---
**Next →** `23_01_huggingface_pipelines_and_transfer_learning.ipynb` — NLP Masterclass 06: HuggingFace Pipelines & Transfer Learning